# tuned - stage the base model as reusable notebook output

One-time run on **CPU (Accelerator: None)**, Internet **On** - burns ZERO GPU quota.
Downloads the pinned `unsloth/Qwen3-8B-unsloth-bnb-4bit` snapshot into this
notebook's output as a flat file tree. Afterwards, in `kaggle_smoke`:
**+ Add Input -> Your Work -> this notebook** - its output mounts under
`/kaggle/input/.../qwen3-8b-staged/` and the training notebook's model cell
auto-detects it (verifying `REVISION.txt` against the config pin) and skips the
hub download entirely.

Re-run only if the pin in `training/configs/law_v1_8b_ddp.yaml` changes - the training
notebook ignores a stale snapshot and falls back to the hub download.

In [ ]:
# Keep REPO/REVISION == training/configs/law_v1_8b_ddp.yaml model pin (tripwire-tested).
import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
REVISION = "62efd7f9d748e394734a7adae2adf96e13a2abc8"
OUT = Path("/kaggle/working/qwen3-8b-staged")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"    # present-and-0: zoo force-enables when absent
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Optional auth via the notebook's HF_TOKEN secret (Add-ons -> Secrets, attach
# to this notebook): higher rate limits / no anonymous-throttling warning. The
# repo is public, so a missing secret just means anonymous - never a failure.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle secret (not printed).")
except Exception as exc:
    print(f"no HF_TOKEN secret ({type(exc).__name__}) - continuing anonymously (public repo)")

# local_dir on hub 1.x writes REAL files (no symlink tree) - exactly the flat
# layout transformers/unsloth can load by path. Bounded attempts with resume:
# a stalled transfer is killed and retried, and retries only fetch what's left.
_code = (
    "from huggingface_hub import snapshot_download\n"
    f"p = snapshot_download({REPO!r}, revision={REVISION!r}, local_dir={str(OUT)!r})\n"
    "print('snapshot at', p)\n"
)
for attempt in (1, 2, 3):
    _t0 = time.time()
    try:
        r = subprocess.run([sys.executable, "-u", "-c", _code], timeout=30 * 60,
                           env={**os.environ, "HF_XET_HIGH_PERFORMANCE": "1"})
    except subprocess.TimeoutExpired:
        print(f"attempt {attempt}: stalled past 30 min - killed; retrying resumes the remainder")
        continue
    if r.returncode == 0:
        print(f"downloaded in {time.time() - _t0:.0f}s (attempt {attempt})")
        break
    print(f"attempt {attempt}: exit {r.returncode} - retrying")
else:
    raise SystemExit("download failed after 3 attempts")

shutil.rmtree(OUT / ".cache", ignore_errors=True)  # hub bookkeeping, not model files
(OUT / "REVISION.txt").write_text(f"{REPO} {REVISION}\n", encoding="utf-8")

files = sorted(p.name for p in OUT.iterdir())
total = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e9
print(f"{len(files)} entries, {total:.2f} GB total")
print("\n".join(files))
assert (OUT / "config.json").is_file(), "config.json missing - incomplete snapshot"
assert any(f.endswith(".safetensors") for f in files), "no safetensors - incomplete snapshot"
print("STAGED OK - attach this notebook's output as an Input to kaggle_smoke")